In [0]:
"""
Data Quality - Extract Faulty Records Task
Filters rows that failed at least one check and enriches them with:
  - original column values (from _input table)
  - individual check results interleaved column by column
  - failed_checks list and failure_count summary
Output schema per data column:
  colX | colX_<dim>_check | colX_<dim2>_check | colY | colY_<dim>_check | ...
"""
from pyspark.sql import functions as F
import re
import json

# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")

PROJECT_ID     = dbutils.widgets.get("project_id")
PROJECT_NAME   = dbutils.widgets.get("project_name")
JOB_NAME       = dbutils.widgets.get("job_name")
CATALOG        = "dbr_uat01_gdm"
SCHEMA         = "db_gdm_dq_results"
RESULTS_SCHEMA = "db_gdm_dq_results"

# Sanitize names for table naming
def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).lower()

project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean     = sanitize_name(JOB_NAME)      if JOB_NAME      else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Extract Faulty Records ===")
print(f"Project: {PROJECT_NAME} (ID: {PROJECT_ID})")
print(f"Job: {JOB_NAME}")
print(f"Results Schema: {RESULTS_SCHEMA}")

# ── Table names ────────────────────────────────────────────────────────────────
input_table  = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_input"
agg_table    = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_checks_aggregated"
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_dq_faulty_records"

# ── Read source tables ─────────────────────────────────────────────────────────
print(f"\nReading input from: {input_table}")
df_input = spark.table(input_table)

print(f"Reading aggregated checks from: {agg_table}")
df_agg = spark.table(agg_table)

total_rows = df_agg.count()
print(f"Total rows in aggregated checks: {total_rows}")

# ── Identify check columns ─────────────────────────────────────────────────────
check_columns = [col for col in df_agg.columns if col.endswith("_check")]
print(f"Found {len(check_columns)} check columns: {check_columns}")

# ── Filter faulty rows (at least one check = 0) ────────────────────────────────
check_array   = F.array(*[F.col(c) for c in check_columns])
has_failure   = F.array_contains(check_array, 0)
df_faulty_agg = df_agg.filter(has_failure)

faulty_count = df_faulty_agg.count()
print(f"Found {faulty_count} rows with at least one failed check")

if faulty_count > 0:
    # ── Build failed_checks list and failure_count ─────────────────────────────
    failed_checks_expr = F.array(*[
        F.when(F.col(col) == 0, F.lit(col.replace("_check", ""))).otherwise(None)
        for col in check_columns
    ])
    df_faulty_agg = (
        df_faulty_agg
        .withColumn("failed_checks_temp", failed_checks_expr)
        .withColumn("failed_checks", F.expr("filter(failed_checks_temp, x -> x is not null)"))
        .drop("failed_checks_temp")
        .withColumn("failure_count", F.size("failed_checks"))
    )

    # ── Join with original input to get source values ──────────────────────────
    # Drop execution_timestamp from input to avoid duplicate column after join
    input_cols_no_ts = [c for c in df_input.columns if c != "execution_timestamp"]
    df_input_clean   = df_input.select(*input_cols_no_ts)

    df_joined = df_faulty_agg.join(df_input_clean, on="row_id", how="left")

    # ── Build interleaved column order ─────────────────────────────────────────
    # Original data columns (exclude row_id and execution_timestamp)
    input_data_cols = [c for c in df_input.columns if c not in ("row_id", "execution_timestamp")]

    ordered_cols = ["row_id"]
    for orig_col in input_data_cols:
        ordered_cols.append(orig_col)
        # Find all check columns that belong to this original column
        matching_checks = [
            chk for chk in check_columns
            if chk.startswith(f"{orig_col}_") and chk.endswith("_check")
        ]
        ordered_cols.extend(sorted(matching_checks))

    # Append summary columns at the end
    ordered_cols += ["failed_checks", "failure_count", "execution_timestamp"]

    # Keep only columns that actually exist in the joined dataframe
    available_cols  = set(df_joined.columns)
    final_cols      = [c for c in ordered_cols if c in available_cols]

    df_faulty_final = df_joined.select(*final_cols)

    print(f"\nFinal schema ({len(final_cols)} columns):")
    for c in final_cols:
        print(f"  {c}")

    # ── Write to Unity Catalog ─────────────────────────────────────────────────
    print(f"\nWriting to: {output_table}")
    df_faulty_final.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(output_table)

    print(f"✅ Successfully wrote {faulty_count} faulty records to {output_table}")

    # ── Display statistics ─────────────────────────────────────────────────────
    print("\nFailure distribution:")
    df_faulty_final.groupBy("failure_count") \
        .count() \
        .orderBy("failure_count") \
        .show()

    print("\nSample faulty records (first 5 rows):")
    df_faulty_final.show(5, truncate=True)

else:
    print("✅ No faulty records found - all checks passed!")